# Analyse your saved anime forum comments
This notebook reads individual JSON checkpoint files from Google Drive and generates new VADER sentiment results. It does not scrape, clone a repository, or require an API key. CPU is sufficient. Run all cells in order.

In [ ]:
import sys, subprocess, json
from pathlib import Path
from datetime import datetime, timezone
subprocess.run([sys.executable, "-m", "pip", "install", "vaderSentiment"], check=True)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/AnimeForum/checkpoints")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Checkpoint folder missing: {DATA_DIR}. Check which Google account you mounted.")
print("Found", len(list(DATA_DIR.glob("*.json"))), "JSON checkpoint files")

In [ ]:
TITLES = {61469: 'Steel Ball Run: JoJo no Kimyou na Bouken', 60636: 'Bleach: Sennen Kessen-hen - Kashin-tan', 21: 'One Piece', 50250: 'Chiikawa', 61607: 'Shiguang Dailiren III', 63832: 'Seihantai na Kimi to Boku 2nd Season', 55809: 'Xian Ni', 60988: 'Tian Guan Cifu Short Films', 51039: 'Doupo Cangqiong: Nian Fan', 56524: 'Tunshi Xingkong 4th Season'}
LIMIT = 5000
analyzer = SentimentIntensityAnalyzer()
summary = {"source": "AniList and MyAnimeList saved forum comments", "limit_per_anime": LIMIT, "model": "VADER", "labels": "Predicted sentiment; no human ground-truth labels", "updated": datetime.now(timezone.utc).isoformat(), "anime": []}
for anime_id, title in TITLES.items():
    source_counts = {}
    counts = {"Positive": 0, "Neutral": 0, "Negative": 0}
    seen = set()
    statuses = {}
    for source, filename in [("AniList", f"{anime_id}.json"), ("MyAnimeList", f"{anime_id}-mal.json")]:
        path = DATA_DIR / filename
        source_counts[source] = 0
        if not path.exists():
            statuses[source] = "not collected"
            continue
        checkpoint = json.loads(path.read_text())
        statuses[source] = "complete" if checkpoint.get("complete") else "partial checkpoint"
        if checkpoint.get("error"):
            statuses[source] = str(checkpoint["error"])
        for comment in checkpoint.get("comments", []):
            if sum(counts.values()) >= LIMIT:
                break
            key = (source, str(comment["id"]))
            text = comment.get("text", "").strip()
            if key in seen or not text:
                continue
            seen.add(key)
            score = analyzer.polarity_scores(text)["compound"]
            label = "Positive" if score >= 0.05 else "Negative" if score <= -0.05 else "Neutral"
            counts[label] += 1
            source_counts[source] += 1
    total = sum(counts.values())
    complete = total == LIMIT or all(status == "complete" for status in statuses.values())
    summary["anime"].append({"title": title, "anime_id": anime_id, "comments": total, "counts": counts, "source_counts": source_counts, "source_status": statuses, "complete": complete, "error": None, "sample_status": "no discussions found" if not total else "small sample" if total < 100 else "sufficient sample"})
    print(f"{title}: {total:,} comments analysed")
if not sum(a["comments"] for a in summary["anime"]):
    raise ValueError("No raw comments found. Check DATA_DIR and the checkpoint JSON files.")

In [ ]:
import pandas as pd
display(pd.DataFrame([{ "Anime": a["title"], "Comments": a["comments"], **a["counts"]} for a in summary["anime"]]))
print("Total:", sum(a["comments"] for a in summary["anime"]))
output = Path("/content/forum.json")
output.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
Path("/content/drive/MyDrive/AnimeForum/forum-colab-results.json").write_text(output.read_text())

In [ ]:
from google.colab import files
files.download(str(output))